In [1]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd
import shutil

In [14]:
#这段代码是把一块下好的AIA不同波段分出来
list1=glob.glob(r'C:\Learning\PHD2nd\sunspotscar\data\to_be_exam\20130411_065500_UTC\JSOC_20260827_001083\*.fits')
list1.sort()
# print(list1[0][-19:-16])
for i in range(len(list1)):
    if list1[i][-19:-16]=='131':
        shutil.copy(list1[i],r'C:\Learning\PHD2nd\sunspotscar\data\to_be_exam\20130411_065500_UTC\131')
    if list1[i][-19:-16]=='211':
        shutil.copy(list1[i],r'C:\Learning\PHD2nd\sunspotscar\data\to_be_exam\20130411_065500_UTC\211')
    if list1[i][-19:-16]=='304':
        shutil.copy(list1[i],r'C:\Learning\PHD2nd\sunspotscar\data\to_be_exam\20130411_065500_UTC\304')

In [3]:
#裁剪目标区域并进行旋转对齐
root_dir=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam'
file_time=os.listdir(root_dir)
# df=pd.read_excel('../dataset/M级耀斑.xlsx')
# #df.drop(df.index[8:16], inplace=True)
# x_range=df['X, arcsec'].values
# y_range=df['Y, arcsec'].values
x_range=[-147.4]
y_range=[215.6]
i=0
#passbands=['131','211','304']
#passbands=['hmi.Ic_45s']
passbands=['1600']
width=800

for j in passbands:
    file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
    os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
    sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
    for k in range(len(file_name)):
        print(k)
        if k==0:
            map0=sunpy.map.Map(file_name[k])
            bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
            tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
            sub_map0=map0.submap(bl1,top_right=tr1)
            sub_map0.meta.pop('BLANK', None)
            sub_map0.save(os.path.join(sub_file_path,file_name[k][-38:]),overwrite=True)
        else:
            map1=sunpy.map.Map(file_name[k])
            with propagate_with_solar_surface():
                rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
            for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                        'bunit', 'exptime']:
                if key in map1.meta:
                    rot_map1.meta[key] = map1.meta[key]

            rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
            rot_map1.meta.pop('BLANK', None)
            rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

# #裁剪Br
# for i in range(59,100):
#     sub_file_path=os.path.join(root_dir,file_time[i],'hmi.B_720s')
#     try:
#         map0=sunpy.map.Map(sub_file_path+'\\'+'Br.fits')
#         bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
#         tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
#         sub_map=map0.submap(bl1,top_right=tr1)
#         sub_map.meta.pop('BLANK', None)
#         sub_map.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
#     except:
#         print(file_time[i]+'hmi.B_720s'+'Br.fits'+'不存在')

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125


In [ ]:
#裁剪目标区域并进行旋转对齐
root_dir=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam'
file_time=os.listdir(root_dir)
# df=pd.read_excel('../dataset/M级耀斑.xlsx')
# #df.drop(df.index[8:16], inplace=True)
# x_range=df['X, arcsec'].values
# y_range=df['Y, arcsec'].values
# x_range=[-147.4]
# y_range=[215.6]
i=0
#passbands=['131','211','304']
passbands=['hmi.Ic_45s']
#'hmi.Ic_45s'
#ref_map=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\131sub_map\aia.lev1_euv_12s.2013-04-11T070510Z.131.image_lev1.fits')
#ref_map=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\M\20130411_065500_UTC\131sub_map\aia.lev1_euv_12s.2013-04-11T062510Z.131.image_lev1.fits')
ref_map=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\M\20130411_065500_UTC\hmi.B_720s\Br_sub.fits')
for j in passbands:
    file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
    os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
    sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
    for k in range(len(file_name)):
        map1=sunpy.map.Map(file_name[k])
        
        with propagate_with_solar_surface():
            rot_map1=map1.reproject_to(ref_map.wcs,preserve_date_obs=True)
        for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                    'bunit', 'exptime']:
            if key in map1.meta:
                rot_map1.meta[key] = map1.meta[key]

        rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
        rot_map1.meta.pop('BLANK', None)
        if j!='hmi.Ic_45s':
            rot_map1.save(os.path.join(sub_file_path, 'aia.lev1_euv_12s.'+file_name[k][-38:]), overwrite=True)
        else:
            rot_map1=rot_map1.rotate()
            rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

# #裁剪Br
# for i in range(59,100):
#     sub_file_path=os.path.join(root_dir,file_time[i],'hmi.B_720s')
#     try:
#         map0=sunpy.map.Map(sub_file_path+'\\'+'Br.fits')
#         bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
#         tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
#         sub_map=map0.submap(bl1,top_right=tr1)
#         sub_map.meta.pop('BLANK', None)
#         sub_map.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
#     except:
#         print(file_time[i]+'hmi.B_720s'+'Br.fits'+'不存在')

In [2]:
#裁剪目标区域并进行旋转对齐
root_dir=r'D:\C'
file_time=os.listdir(root_dir)

#旋转裁剪对齐Br/inclination
for i in range(len(file_time)):
    try:
        ref_131=glob.glob(os.path.join(root_dir,file_time[i],'131sub_map','*.fits'))[0]
        ref_sub_map=sunpy.map.Map(ref_131)
        map1=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc.fits'))
        with propagate_with_solar_surface():
            rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
        for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                    'bunit', 'exptime']:
            if key in map1.meta:
                rot_map1.meta[key] = map1.meta[key]
        rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
        rot_map1.meta.pop('BLANK', None)
        rot_map1.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc_sub.fits'), overwrite=True)
    except:
        print(file_time[i]+'hmi.B_720s'+'inclination_calc_sub.fits'+'保存失败')


20150313_073200_UTChmi.B_720sinclination_calc_sub.fits保存失败
20150916_190800_UTChmi.B_720sinclination_calc_sub.fits保存失败
20151228_001800_UTChmi.B_720sinclination_calc_sub.fits保存失败


In [1]:
from pathlib import Path
import shutil

src_root = Path(r"F:\C")
dst_root = Path(r"E:\Learning\PHD3rd\sunspotscar\data\C")
names = ["131sub_map", "211sub_map", "304sub_map", "hmi.B_720s", "hmi.Ic_45ssub_map"]

if not src_root.is_dir():
    raise FileNotFoundError(f"源文件夹不存在：{src_root}")

for event in sorted(src_root.iterdir())[18:]:
    if not event.is_dir():
        continue
    for name in names:
        src = event / name
        dst = dst_root / event.name / name
        if not src.is_dir():
            print(f"缺失，跳过：{src}")
            continue
        if dst.exists():
            print(f"目标已存在，跳过：{dst}")
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        print(f"已移动：{event.name} / {name}")

print("处理完成")

目标已存在，跳过：E:\Learning\PHD3rd\sunspotscar\data\C\20130518_033000_UTC\131sub_map
已移动：20130518_033000_UTC / 211sub_map
已移动：20130518_033000_UTC / 304sub_map
已移动：20130518_033000_UTC / hmi.B_720s
已移动：20130518_033000_UTC / hmi.Ic_45ssub_map
已移动：20130520_145900_UTC / 131sub_map
已移动：20130520_145900_UTC / 211sub_map
已移动：20130520_145900_UTC / 304sub_map
已移动：20130520_145900_UTC / hmi.B_720s
已移动：20130520_145900_UTC / hmi.Ic_45ssub_map
已移动：20130520_162000_UTC / 131sub_map
已移动：20130520_162000_UTC / 211sub_map
已移动：20130520_162000_UTC / 304sub_map
已移动：20130520_162000_UTC / hmi.B_720s
已移动：20130520_162000_UTC / hmi.Ic_45ssub_map
已移动：20130628_033100_UTC / 131sub_map
已移动：20130628_033100_UTC / 211sub_map
已移动：20130628_033100_UTC / 304sub_map
已移动：20130628_033100_UTC / hmi.B_720s
已移动：20130628_033100_UTC / hmi.Ic_45ssub_map
已移动：20130707_005300_UTC / 131sub_map
已移动：20130707_005300_UTC / 211sub_map
已移动：20130707_005300_UTC / 304sub_map
已移动：20130707_005300_UTC / hmi.B_720s
已移动：20130707_005300_UTC / hmi.Ic_45ssub_map